In [13]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import urllib.parse
import matplotlib.pyplot as plt


from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "pass909275"

db = AnimalShelter(username, password)

# Initial "Retrieve All" query for unfiltered view
df = pd.DataFrame.from_records(db.read({}))

# Clean the data: Remove MongoDB ObjectID to prevent DataTable crashes
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' 
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
    # Header with Logo and Student Identifier
    html.Div([
        html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), 
                 style={'height':'150px', 'width':'auto', 'float':'left'}),
        html.Center(html.B(html.H1('SNHU CS-340 Dashboard'))),
        html.Center(html.P("Unique Identifier: Rudolph travers - 2026-02-21")), 
    ], style={'display': 'flow-root', 'padding': '10px'}),
    
    html.Hr(),
    
    # Interactive Filtering Options
    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'WR'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'MR'},
                {'label': 'Disaster or Individual Tracking', 'value': 'DT'},
                {'label': 'Reset', 'value': 'RESET'}
            ],
            value='RESET',
            labelStyle={'display': 'inline-block', 'padding': '10px'}
        ),
    ]),
    
    html.Hr(),
    
    # Interactive Data Table
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        # Interactive features for user-friendliness
        page_size=10,           
        sort_action="native",   
        filter_action="native", 
        row_selectable="single", 
        selected_rows=[0],       
        style_table={'overflowX': 'auto'}, 
        style_cell={
            'textAlign': 'left',
            'padding': '5px',
            'whiteSpace': 'normal',
            'height': 'auto',
        },
    ),
    
    html.Br(),
    html.Hr(),
    
    # Side-by-Side Visualization Row
    html.Div(className='row', 
        style={'display' : 'flex'}, 
             children=[
        html.Div(
            id='graph-id', 
        className='col s12 m6', 
        style={'flex': '1'}),
        html.Div(id='map-id',
            className='col s12 m6', 
                 style={'flex': '1'})
    ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
    )
              
def update_dashboard(filter_type):
    # Specific queries based on Rescue Type Specifications
    if filter_type == 'WR': # Water Rescue
        query = {"animal_type": "Dog", "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]}, "sex_upon_outcome": "Intact Female", "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}}
    elif filter_type == 'MR': # Mountain/Wilderness Rescue
        query = {"animal_type": "Dog", "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]}, "sex_upon_outcome": "Intact Male", "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}}
    elif filter_type == 'DT': # Disaster/Tracking
        query = {"animal_type": "Dog", "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]}, "sex_upon_outcome": "Intact Male", "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}}
    else: # Reset/All
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
    return dff.to_dict('records')
              
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    if viewData is None:
        return []
    dff = pd.DataFrame.from_dict(viewData)
    return [
        dcc.Graph(figure=px.pie(dff, names='breed', title='Preferred Dog Breeds Representation'))
    ]

# Callback to update the Map based on selected row
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None or not index:
        return []
    
    dff = pd.DataFrame.from_dict(viewData)
    row = index[0]
        
    return [
        dl.Map(style={'width': '100%', 'height': '500px'}, center=[30.75, -97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[dff.iloc[row, 13], dff.iloc[row, 14]], children=[
                dl.Tooltip(dff.iloc[row, 4]), # Breed
                dl.Popup([html.H2("Animal Name"), html.P(dff.iloc[row, 9])]) # Name
            ])
        ])
    ]

# Run app on a unique port
app.run_server(port=8055)

Dash app running on https://shorttrinity-borisship-3000.codio.io/proxy/8055/
